# ms-DPG tutorial: \emph{emotions}

This tutorial fits a native multi-output random forest and reads ms-DPG at two levels: the SPF matrix identifies an ordered label pair; the pair explanation names shared predicates, their lifts, and a contracted view.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from msdpg import MSDPG

## Load the pinned OpenML record

The target columns are converted to binary indicators. The feature matrix retains numeric audio descriptors.

In [ ]:
data = fetch_openml(name="emotions", version=3, as_frame=True, parser="auto")
frame = data.frame
target_names = list(data.target_names)
X = frame.drop(columns=target_names).select_dtypes(include=["number"]).fillna(0).to_numpy(float)
Y = frame[target_names].astype("category").apply(lambda c: c.cat.codes).to_numpy()
Y = (Y > 0).astype(int)
X.shape, Y.shape, target_names

## Fit the native forest and obtain SPF

`fit` only inspects the fitted forest and the training matrix. Quantile edges are computed from the latter.

In [ ]:
forest = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(X, Y)
exp = MSDPG(matching="quantile", bins=32).fit(
    forest, X, label_names=target_names, feature_names=list(frame.drop(columns=target_names).select_dtypes(include=["number"]).columns)
).explain()
exp.top_pairs(5)

In [ ]:
ax = exp.plot_spf()
ax.set_title("Shared Predicate Flow")

## Inspect one ordered pair

The `shared` table reports directional predicates. A lift above one means the corresponding label is enriched among paths traversing that predicate. SPF describes fitted decision structure, not statistical label dependence.

In [ ]:
pair = exp.pair("sad.lonely", "quiet.still")
print("SPF:", pair.spf)
pair.shared.head(10)

In [ ]:
pair.plot_lifts()
view = pair.contracted(n_nodes=14)
print("sink coverage:", view.coverage)
view.to_graphviz("emotions_contracted")

## Binary Relevance comparison

The same API accepts a fitted `MultiOutputClassifier` of random forests. BR supports SPF, but lifts and contracted views are intentionally unavailable because a BR trace belongs to only one label forest.